# Hybrid IDS SDS Shapes: Exact MFV-NSA + Fuzzy Membership + Adaptive Memory CSA

This notebook preserves the original FinalV2 pipeline **without fivefold cross-validation**, while replacing the former **Multi-Objective V-Detector NSA + Fuzzy NSA** block with the exact Section **3.4.2 Multi-Constrained Fuzzy V-Detector NSA with Fuzzy Membership (MFV-NSA)** implementation from the attached manuscript.

Included:
1. Exact MFV-NSA Stages 1–5  
2. Algorithms 1–5 implemented step by step  
3. Table 5 membership behavior and Table 6 14-dimensional MFV-NSA features  
4. Adaptive Memory CSA retained in the original pipeline position  
5. Triangle shape with Cross, Ring, and Pentagram  
6. 10,000 samples per shape dataset  
7. Self / non-self split  
8. Coverage heat-map and hole-map visualizations  
9. Final train/test evaluation only  
10. Existing SDS sensitivity and plotting blocks retained


In [ ]:
# ============================================================
# 1. Imports and global configuration
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.path import Path
from sklearn.neighbors import BallTree
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

N_PER_SHAPE = 10_000        # 10,000 samples for each shape
GRID_SIZE = 120             # heat-map resolution
N_DETECTORS = 120           # final detector memory size per shape
N_CANDIDATES = 1200         # candidate detector centers
MAX_RADIUS = 0.12
MIN_RADIUS = 0.008
SELF_MARGIN = 0.003

SHAPES = ["Cross", "Ring", "Pentagram", "Triangle"]

plt.rcParams["figure.dpi"] = 120


In [ ]:
# ============================================================
# 2. Shape geometry functions
#    self = inside shape
#    non-self = outside shape
# ============================================================

def cross_mask(points):
    """Cross self-region."""
    x, y = points[:, 0], points[:, 1]
    vertical = (x >= 0.40) & (x <= 0.60) & (y >= 0.15) & (y <= 0.85)
    horizontal = (x >= 0.15) & (x <= 0.85) & (y >= 0.40) & (y <= 0.60)
    return vertical | horizontal


def ring_mask(points):
    """Ring/annulus self-region."""
    x, y = points[:, 0], points[:, 1]
    r = np.sqrt((x - 0.5) ** 2 + (y - 0.5) ** 2)
    return (r >= 0.13) & (r <= 0.33)


def regular_star_vertices(center=(0.5, 0.5), outer_r=0.36, inner_r=0.145, n_points=5):
    """Create a pentagram/star polygon."""
    cx, cy = center
    vertices = []
    start_angle = np.pi / 2
    for i in range(2 * n_points):
        angle = start_angle + i * np.pi / n_points
        r = outer_r if i % 2 == 0 else inner_r
        vertices.append((cx + r * np.cos(angle), cy + r * np.sin(angle)))
    return np.array(vertices)


STAR_VERTICES = regular_star_vertices()


def pentagram_mask(points):
    """Pentagram self-region using a star polygon."""
    return Path(STAR_VERTICES).contains_points(points)


TRIANGLE_VERTICES = np.array([
    [0.50, 0.86],
    [0.14, 0.18],
    [0.86, 0.18]
])


def triangle_mask(points):
    """Triangle self-region."""
    return Path(TRIANGLE_VERTICES).contains_points(points)


def shape_mask(points, shape_name):
    if shape_name == "Cross":
        return cross_mask(points)
    if shape_name == "Ring":
        return ring_mask(points)
    if shape_name == "Pentagram":
        return pentagram_mask(points)
    if shape_name == "Triangle":
        return triangle_mask(points)
    raise ValueError(f"Unknown shape: {shape_name}")


In [ ]:
# ============================================================
# 3. Generate 10,000-sample dataset for each shape
# ============================================================

def generate_shape_dataset(shape_name, n_samples=N_PER_SHAPE, seed=RANDOM_STATE):
    local_rng = np.random.default_rng(seed + abs(hash(shape_name)) % 10_000)
    points = local_rng.random((n_samples, 2))
    self_flag = shape_mask(points, shape_name).astype(int)

    df = pd.DataFrame({
        "x": points[:, 0],
        "y": points[:, 1],
        "shape": shape_name,
        "shape_id": SHAPES.index(shape_name),
        "self_label": self_flag,              # 1 = self/normal, 0 = non-self/attack
        "class_name": np.where(self_flag == 1, "Self", "Non-Self")
    })

    # Additional simple geometric features
    df["dist_center"] = np.sqrt((df["x"] - 0.5) ** 2 + (df["y"] - 0.5) ** 2)
    df["angle"] = np.arctan2(df["y"] - 0.5, df["x"] - 0.5)
    df["xy"] = df["x"] * df["y"]
    df["x2"] = df["x"] ** 2
    df["y2"] = df["y"] ** 2
    return df


all_dfs = []
for shp in SHAPES:
    all_dfs.append(generate_shape_dataset(shp, N_PER_SHAPE))

dataset = pd.concat(all_dfs, ignore_index=True)

print("Full dataset shape:", dataset.shape)
print(dataset.groupby(["shape", "class_name"]).size())
dataset.head()


In [ ]:
# ============================================================
# 4. Self / non-self split
# ============================================================

self_df = dataset[dataset["self_label"] == 1].copy()
nonself_df = dataset[dataset["self_label"] == 0].copy()

print("Total samples:", len(dataset))
print("Self samples:", len(self_df))
print("Non-self samples:", len(nonself_df))
print("\nPer-shape self/non-self distribution:")
display(dataset.groupby(["shape", "class_name"]).size().unstack(fill_value=0))


In [ ]:
# ============================================================
# 5. Plot shape datasets
# ============================================================

def plot_shape_scatter(df, shape_name, title=None):
    sub = df[df["shape"] == shape_name]
    plt.figure(figsize=(4.6, 4.6))
    colors = np.where(sub["self_label"].values == 1, "tab:blue", "tab:orange")
    plt.scatter(sub["x"], sub["y"], c=colors, s=4, alpha=0.65, linewidths=0)
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.gca().set_aspect("equal", adjustable="box")
    plt.title(title or f"{shape_name}: Self and Non-Self Dataset", fontweight="bold")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.grid(alpha=0.15)
    plt.show()


# Pentagram figure similar to the reference
plot_shape_scatter(dataset, "Pentagram", "(c) Pentagram")


In [ ]:
# ============================================================
# 5A. Plot shape for all SDS datasets
#     This cell keeps the original code unchanged and adds
#     one complete visualization for Cross, Ring, Pentagram, Triangle.
# ============================================================

def plot_all_shape_datasets(df, shapes=SHAPES):
    n_shapes = len(shapes)
    fig, axes = plt.subplots(1, n_shapes, figsize=(4.2 * n_shapes, 4.2), sharex=True, sharey=True)

    if n_shapes == 1:
        axes = [axes]

    for ax, shape_name in zip(axes, shapes):
        sub = df[df["shape"] == shape_name]
        self_part = sub[sub["self_label"] == 1]
        nonself_part = sub[sub["self_label"] == 0]

        # Non-self/attack points outside the shape
        ax.scatter(
            nonself_part["x"], nonself_part["y"],
            s=3, alpha=0.55, linewidths=0,
            label="Non-self"
        )

        # Self/normal points inside the shape
        ax.scatter(
            self_part["x"], self_part["y"],
            s=3, alpha=0.70, linewidths=0,
            label="Self"
        )

        ax.set_title(f"{shape_name}", fontweight="bold")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_aspect("equal", adjustable="box")
        ax.grid(alpha=0.15)
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.legend(loc="upper right", fontsize=8, frameon=True)

    fig.suptitle("SDS Shape Datasets: Self vs Non-Self", fontweight="bold", y=1.03)
    plt.tight_layout()
    plt.show()


plot_all_shape_datasets(dataset, SHAPES)


In [ ]:

# ============================================================
# 6. EXACT PAPER IMPLEMENTATION:
#    3.4.2. Multi-Constrained Fuzzy V-Detector NSA with
#    Fuzzy Membership (MFV-NSA)
#
#    Stage 1 — Self-Set Construction            -> Algorithm 1
#    Stage 2 — Adaptive Self-Scale Estimation   -> Algorithm 1
#    Stage 3 — Multi-Constrained V-Detectors    -> Algorithm 2
#    Stage 4 — Fuzzy Anomaly Score              -> Algorithm 3
#    Stage 5 — 14-D MFV-NSA Feature Assembly    -> Algorithm 4
#    Cross-fitted MFV-NSA pipeline               -> Algorithm 5
#
# IMPORTANT:
# - The original FinalV2 cell order/pipeline is preserved.
# - The previous Pareto/multi-objective scoring has been removed.
# - Detector acceptance now uses ONLY the two constraints stated
#   in Section 3.4.2: distance-to-self + detector separation.
# ============================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

# ------------------------------------------------------------
# Exact Section 3.4.2 constants
# ------------------------------------------------------------
MFV_M = 8000
MFV_RHO = 95.0
MFV_EPS_R = 1e-12

MFV_R_MIN = 0.15
MFV_R_MAX = 5.0
MFV_DELTA = 1e-6
MFV_BETA = 0.30
MFV_N_MAX = 600

MFV_GAMMA = 6.0
MFV_W_D = 0.60
MFV_W_A = 0.40
MFV_EPS = 1e-9
MFV_K = 10

# Algorithm 5 contains N_cand as an input but the manuscript does not
# prescribe a single numerical value. Keep the FinalV2 candidate budget.
MFV_N_CAND = N_CANDIDATES


def make_grid(grid_size=GRID_SIZE):
    xs = np.linspace(0, 1, grid_size)
    ys = np.linspace(0, 1, grid_size)
    xx, yy = np.meshgrid(xs, ys)
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    return xs, ys, xx, yy, grid_points


def detector_overlap_penalty(center, radius, chosen_centers, chosen_radii):
    """
    Legacy helper retained because later FinalV2 analysis/CSA cells call it.
    It is NOT used by the exact MFV-NSA detector acceptance rule below.
    """
    if len(chosen_centers) == 0:
        return 0.0
    centers = np.asarray(chosen_centers, dtype=float)
    radii = np.asarray(chosen_radii, dtype=float)
    d = np.linalg.norm(centers - center, axis=1)
    overlap = np.maximum(0.0, radius + radii - d)
    return float(np.mean(overlap / (radius + radii + 1e-12)))


# ============================================================
# Algorithm 1
# Training-Only Self-Set Construction and Adaptive Boundary
# Estimation
# ============================================================
def algorithm1_self_set_and_boundary(
    X_fit,
    y_bin,
    M=MFV_M,
    rho=MFV_RHO,
    seed=RANDOM_STATE,
    eps_r=MFV_EPS_R
):
    """
    Exact implementation of Algorithm 1 from Section 3.4.2.

    Parameters
    ----------
    X_fit : array, shape (n_samples, n_features)
        Current training partition.
    y_bin : array, shape (n_samples,)
        Paper convention: 0 = benign/self, 1 = attack/non-self.
    """
    X_fit = np.asarray(X_fit, dtype=float)
    y_bin = np.asarray(y_bin, dtype=int)

    # Step 1: retain benign training samples only.
    S = X_fit[y_bin == 0].copy()

    # Step 2: remove exact duplicates.
    if len(S) > 0:
        S = np.unique(S, axis=0)

    # Steps 3–5: at least two distinct self samples are required.
    if len(S) < 2:
        raise ValueError("Insufficient benign training samples for MFV-NSA self-set construction.")

    # Steps 6–8: fixed-seed uniform subsampling without replacement.
    if len(S) > M:
        local_rng = np.random.default_rng(seed)
        idx = local_rng.choice(len(S), size=M, replace=False)
        S = S[idx]

    # Step 9: spatial index fitted using self-set only.
    T_S = BallTree(S)

    # Steps 10–14: nearest distinct-self distances.
    nn_dist, _ = T_S.query(S, k=2)
    Delta = nn_dist[:, 1]  # k=1 is the query itself (distance 0)

    # Step 15: adaptive boundary radius.
    r_boundary = max(float(np.percentile(Delta, rho)), float(eps_r))

    # Step 16: return S, boundary radius, spatial index.
    return S, r_boundary, T_S


# ============================================================
# Training-derived candidate pool (Algorithm 5: SAMPLECANDIDATES)
# ============================================================
def sample_candidates_from_training(X_fit, n_candidates=MFV_N_CAND, seed=RANDOM_STATE):
    """
    Build the candidate pool strictly from the current training feature space.
    No validation/test sample is used.
    """
    X_fit = np.asarray(X_fit, dtype=float)
    if len(X_fit) == 0:
        return np.empty((0, X_fit.shape[1] if X_fit.ndim == 2 else 0), dtype=float)

    X_unique = np.unique(X_fit, axis=0)
    local_rng = np.random.default_rng(seed)

    if len(X_unique) <= n_candidates:
        order = local_rng.permutation(len(X_unique))
        return X_unique[order]

    idx = local_rng.choice(len(X_unique), size=n_candidates, replace=False)
    return X_unique[idx]


# ============================================================
# Algorithm 2
# Multi-Constrained Fuzzy V-Detector Generation
# ============================================================
def algorithm2_generate_detectors(
    self_set,
    self_tree,
    candidate_pool,
    r_min=MFV_R_MIN,
    r_max=MFV_R_MAX,
    N_max=MFV_N_MAX,
    beta=MFV_BETA,
    delta=MFV_DELTA
):
    """
    Exact detector-acceptance logic from Algorithm 2.

    Component 1: r_min <= nearest-self distance <= r_max
                 detector radius = d(self) - delta
    Component 2: center separation >= beta * min(r_c, r_k)
    Component 3: accept only when BOTH constraints hold
    """
    P = np.asarray(candidate_pool, dtype=float)
    if P.ndim != 2:
        raise ValueError("candidate_pool must be a 2-D array.")

    accepted_centers = []
    accepted_radii = []

    for c in P:
        # Steps 3–5: stop at the detector budget.
        if len(accepted_centers) >= N_max:
            break

        # Steps 6–10: Component 1 — distance-to-self constraint.
        d_S = float(self_tree.query(c.reshape(1, -1), k=1)[0][0, 0])
        if d_S < r_min or d_S > r_max:
            continue

        r_c = d_S - delta
        if r_c <= 0:
            continue

        # Steps 11–17: Component 2 — detector-separation constraint.
        valid = True
        if accepted_centers:
            D = np.asarray(accepted_centers, dtype=float)
            R = np.asarray(accepted_radii, dtype=float)
            center_dist = np.linalg.norm(D - c, axis=1)
            required_sep = beta * np.minimum(r_c, R)
            if np.any(center_dist < required_sep):
                valid = False

        # Steps 18–20: Component 3 — detector acceptance.
        if valid:
            accepted_centers.append(c.copy())
            accepted_radii.append(float(r_c))

    if accepted_centers:
        centers = np.asarray(accepted_centers, dtype=float)
        radii = np.asarray(accepted_radii, dtype=float)
    else:
        centers = np.empty((0, P.shape[1]), dtype=float)
        radii = np.empty((0,), dtype=float)

    return centers, radii


def _build_detector_tree(centers):
    centers = np.asarray(centers, dtype=float)
    return BallTree(centers) if len(centers) > 0 else None


# ============================================================
# Algorithm 3
# Fuzzy Anomaly Score Computation
# ============================================================
def algorithm3_fuzzy_anomaly_score(
    x,
    self_tree,
    centers,
    radii,
    r_boundary,
    gamma=MFV_GAMMA,
    w_d=MFV_W_D,
    w_a=MFV_W_A,
    eps=MFV_EPS,
    K=MFV_K,
    detector_tree=None
):
    """
    Exact Algorithm 3 for one query x.
    Returns:
        s_fuzzy, mu_dist, mu_det, d_self
    """
    x = np.asarray(x, dtype=float).reshape(1, -1)
    centers = np.asarray(centers, dtype=float)
    radii = np.asarray(radii, dtype=float)

    # Component 1 — distance-based membership.
    d_self = float(self_tree.query(x, k=1)[0][0, 0])
    logit_arg = np.clip(gamma * (d_self - r_boundary), -60.0, 60.0)
    mu_dist = float(1.0 / (1.0 + np.exp(-logit_arg)))

    # Algorithm 3, Steps 3–5.
    if len(centers) == 0:
        return 0.0, mu_dist, 0.0, d_self

    # Component 2 — detector-activation membership.
    Kp = min(int(K), len(centers))
    if detector_tree is None:
        detector_tree = BallTree(centers)

    dists, idx = detector_tree.query(x, k=Kp)
    dists = dists[0]
    idx = idx[0]
    count = int(np.sum(dists <= radii[idx]))
    mu_det = float(count / Kp)

    # Component 3 — weighted harmonic fuzzy fusion.
    s_fuzzy = (w_d + w_a) / (
        w_d / (mu_dist + eps) +
        w_a / (mu_det + eps)
    )
    s_fuzzy = float(min(1.0, s_fuzzy))

    return s_fuzzy, mu_dist, mu_det, d_self


# ============================================================
# Algorithm 4
# MFV-NSA 14-D Feature-Vector Assembly
# ============================================================
MFV_NSA_FEATURE_COLUMNS = [
    "mfv_d_self",          # 1
    "mfv_s_fuzzy",         # 2
    "mfv_mu_dist",         # 3
    "mfv_mu_det",          # 4
    "mfv_d_min",           # 5
    "mfv_d_mean",          # 6
    "mfv_d_std",           # 7
    "mfv_r_min_active",    # 8
    "mfv_r_mean_active",   # 9
    "mfv_r_std_active",    # 10
    "mfv_act_k",           # 11
    "mfv_p_max",           # 12
    "mfv_p_mean",          # 13
    "mfv_margin",          # 14
]


def algorithm4_mfv_nsa_feature_vector(
    x,
    self_tree,
    centers,
    radii,
    r_boundary,
    tau,
    gamma=MFV_GAMMA,
    w_d=MFV_W_D,
    w_a=MFV_W_A,
    eps=MFV_EPS,
    K=MFV_K,
    detector_tree=None
):
    """
    Exact Algorithm 4 for one query x.
    """
    x_arr = np.asarray(x, dtype=float).reshape(1, -1)
    centers = np.asarray(centers, dtype=float)
    radii = np.asarray(radii, dtype=float)

    # Component 1: Core fuzzy features (1–4).
    s_fuzzy, mu_dist, mu_det, d_self = algorithm3_fuzzy_anomaly_score(
        x_arr[0], self_tree, centers, radii, r_boundary,
        gamma=gamma, w_d=w_d, w_a=w_a, eps=eps, K=K,
        detector_tree=detector_tree
    )

    # Component 2: Detector-distance features (5–7).
    if len(centers) > 0:
        Kp = min(int(K), len(centers))
        if detector_tree is None:
            detector_tree = BallTree(centers)
        dists, idx = detector_tree.query(x_arr, k=Kp)
        dists = dists[0]
        idx = idx[0]

        d_min = float(np.min(dists))
        d_mean = float(np.mean(dists))
        d_std = float(np.std(dists))

        # Component 3: Activation-radius features (8–10).
        active_mask = dists <= radii[idx]
        active_idx = idx[active_mask]

        if len(active_idx) > 0:
            Rlist = radii[active_idx]
            r_min_active = float(np.min(Rlist))
            r_mean_active = float(np.mean(Rlist))
            r_std_active = float(np.std(Rlist))
        else:
            r_min_active = r_mean_active = r_std_active = 0.0

        # Component 4: Activation and penetration features (11–13).
        act_k = float(len(active_idx))
        if len(active_idx) > 0:
            active_dists = dists[active_mask]
            penetration = radii[active_idx] - active_dists
            p_max = float(np.max(penetration))
            p_mean = float(np.mean(penetration))
        else:
            p_max = p_mean = 0.0
    else:
        d_min = d_mean = d_std = 0.0
        r_min_active = r_mean_active = r_std_active = 0.0
        act_k = 0.0
        p_max = p_mean = 0.0

    # Component 5: Decision margin and final assembly (14).
    margin = float(s_fuzzy - tau)

    return np.asarray([
        d_self, s_fuzzy, mu_dist, mu_det,
        d_min, d_mean, d_std,
        r_min_active, r_mean_active, r_std_active,
        act_k, p_max, p_mean, margin
    ], dtype=float)


# ------------------------------------------------------------
# Vectorized equivalent of Algorithms 3–4 for the full dataset.
# It preserves the exact formulas while avoiding 40k Python loops.
# ------------------------------------------------------------
def mfv_nsa_features_batch(
    points,
    model_result,
    tau=None,
    gamma=MFV_GAMMA,
    w_d=MFV_W_D,
    w_a=MFV_W_A,
    eps=MFV_EPS,
    K=MFV_K
):
    X = np.asarray(points, dtype=float)
    self_tree = model_result["self_tree"]
    r_boundary = float(model_result["boundary_radius"])

    # Keep the exact Section 3.4.2 detector bank immutable even if
    # later legacy FinalV2 cells manipulate their working copy.
    centers = np.asarray(
        model_result.get("mfv_centers", model_result["centers"]),
        dtype=float
    )
    radii = np.asarray(
        model_result.get("mfv_radii", model_result["radii"]),
        dtype=float
    )

    if tau is None:
        tau = float(model_result.get("threshold", 0.5))

    # Features 1 and 3.
    d_self = self_tree.query(X, k=1)[0][:, 0]
    logit_arg = np.clip(gamma * (d_self - r_boundary), -60.0, 60.0)
    mu_dist = 1.0 / (1.0 + np.exp(-logit_arg))

    n = len(X)

    if len(centers) == 0:
        s_fuzzy = np.zeros(n, dtype=float)
        mu_det = np.zeros(n, dtype=float)

        d_min = np.zeros(n, dtype=float)
        d_mean = np.zeros(n, dtype=float)
        d_std = np.zeros(n, dtype=float)

        r_min_active = np.zeros(n, dtype=float)
        r_mean_active = np.zeros(n, dtype=float)
        r_std_active = np.zeros(n, dtype=float)

        act_k = np.zeros(n, dtype=float)
        p_max = np.zeros(n, dtype=float)
        p_mean = np.zeros(n, dtype=float)
    else:
        detector_tree = BallTree(centers)
        Kp = min(int(K), len(centers))
        dists, idx = detector_tree.query(X, k=Kp)
        neighbor_r = radii[idx]
        active = dists <= neighbor_r

        # Feature 4.
        mu_det = active.mean(axis=1)

        # Feature 2: weighted harmonic fuzzy fusion.
        s_fuzzy = (w_d + w_a) / (
            w_d / (mu_dist + eps) +
            w_a / (mu_det + eps)
        )
        s_fuzzy = np.minimum(1.0, s_fuzzy)

        # Features 5–7.
        d_min = dists.min(axis=1)
        d_mean = dists.mean(axis=1)
        d_std = dists.std(axis=1)

        # Feature 11.
        act_k = active.sum(axis=1).astype(float)

        # Features 8–10.
        active_r = np.where(active, neighbor_r, 0.0)
        active_count = np.maximum(act_k, 1.0)

        r_sum = active_r.sum(axis=1)
        r_mean_active = np.where(act_k > 0, r_sum / active_count, 0.0)

        # min over active detector radii only
        r_min_tmp = np.where(active, neighbor_r, np.inf).min(axis=1)
        r_min_active = np.where(act_k > 0, r_min_tmp, 0.0)

        r_var_num = np.where(
            active,
            (neighbor_r - r_mean_active[:, None]) ** 2,
            0.0
        ).sum(axis=1)
        r_std_active = np.where(
            act_k > 0,
            np.sqrt(r_var_num / active_count),
            0.0
        )

        # Features 12–13.
        penetration = np.where(active, neighbor_r - dists, 0.0)
        p_max = np.where(act_k > 0, penetration.max(axis=1), 0.0)
        p_mean = np.where(
            act_k > 0,
            penetration.sum(axis=1) / active_count,
            0.0
        )

    # Feature 14.
    margin = s_fuzzy - tau

    return np.column_stack([
        d_self, s_fuzzy, mu_dist, mu_det,
        d_min, d_mean, d_std,
        r_min_active, r_mean_active, r_std_active,
        act_k, p_max, p_mean, margin
    ])


def mfv_nsa_score_batch(points, model_result):
    """Exact Stage-4 fuzzy score for a batch of queries."""
    return mfv_nsa_features_batch(points, model_result)[:, 1]


# ============================================================
# Threshold tuning used by Algorithm 5 and the FinalV2 wrapper.
# Only the supplied validation/OOF labels are used.
# ============================================================
def tune_mfv_threshold(y_bin, fuzzy_scores, thresholds=None):
    y_bin = np.asarray(y_bin, dtype=int)
    fuzzy_scores = np.asarray(fuzzy_scores, dtype=float)

    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 91)

    best_tau = 0.5
    best_f1 = -np.inf

    for tau in thresholds:
        pred = (fuzzy_scores >= tau).astype(int)
        score = f1_score(y_bin, pred, average="macro", zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_tau = float(tau)

    return best_tau, float(best_f1)


def _fit_mfv_model(
    X_fit,
    y_bin_fit,
    seed=RANDOM_STATE,
    M=MFV_M,
    rho=MFV_RHO,
    n_candidates=MFV_N_CAND,
    r_min=MFV_R_MIN,
    r_max=MFV_R_MAX,
    N_max=MFV_N_MAX,
    beta=MFV_BETA,
    delta=MFV_DELTA
):
    """Fit Algorithms 1–2 on one training partition."""
    S, r_boundary, T_S = algorithm1_self_set_and_boundary(
        X_fit, y_bin_fit, M=M, rho=rho, seed=seed
    )

    P = sample_candidates_from_training(
        X_fit, n_candidates=n_candidates, seed=seed
    )

    centers, radii = algorithm2_generate_detectors(
        S, T_S, P,
        r_min=r_min,
        r_max=r_max,
        N_max=N_max,
        beta=beta,
        delta=delta
    )

    return {
        "self_set": S,
        "self_tree": T_S,
        "boundary_radius": float(r_boundary),
        "candidate_pool": P,
        "centers": centers,
        "radii": radii,
        "detector_tree": _build_detector_tree(centers)
    }


# ============================================================
# Algorithm 5
# Cross-Fitted MFV-NSA Pipeline
# ============================================================
def algorithm5_cross_fitted_mfv_nsa(
    X_train,
    y_bin_train,
    X_eval,
    groups=None,
    K_cf=5,
    M=MFV_M,
    rho=MFV_RHO,
    N_cand=MFV_N_CAND,
    r_min=MFV_R_MIN,
    r_max=MFV_R_MAX,
    N_max=MFV_N_MAX,
    beta=MFV_BETA,
    delta=MFV_DELTA,
    gamma=MFV_GAMMA,
    w_d=MFV_W_D,
    w_a=MFV_W_A,
    eps=MFV_EPS,
    K=MFV_K,
    seed=RANDOM_STATE
):
    """
    Exact three-phase structure of Algorithm 5.

    Phase 1: each training row receives an out-of-fold 14-D MFV-NSA vector.
    Phase 2: refit the immune model on the complete training partition and
             select tau* from OOF fuzzy scores only.
    Phase 3: transform evaluation samples inductively with the frozen model.

    Paper label convention: 0=benign/self, 1=attack/non-self.
    """
    X_train = np.asarray(X_train, dtype=float)
    y_bin_train = np.asarray(y_bin_train, dtype=int)
    X_eval = np.asarray(X_eval, dtype=float)

    if K_cf < 2:
        raise ValueError("K_cf must be >= 2 for Algorithm 5 cross-fitting.")

    Z_oof = np.zeros((len(X_train), 14), dtype=float)

    # Algorithm 5 specifies GROUPSTRATIFIEDFOLDS. If provenance/group
    # identifiers are supplied, use StratifiedGroupKFold. The SDS FinalV2
    # dataset has no provenance group field, so the no-group fallback is
    # stratified K-fold over the training partition only.
    if groups is not None:
        try:
            from sklearn.model_selection import StratifiedGroupKFold
            splitter = StratifiedGroupKFold(
                n_splits=K_cf,
                shuffle=True,
                random_state=seed
            )
            split_iter = splitter.split(X_train, y_bin_train, groups=np.asarray(groups))
        except ImportError:
            raise ImportError(
                "StratifiedGroupKFold is required when groups are supplied."
            )
    else:
        splitter = StratifiedKFold(
            n_splits=K_cf,
            shuffle=True,
            random_state=seed
        )
        split_iter = splitter.split(X_train, y_bin_train)

    # Phase 1 — Cross-Fitted Training-Feature Generation.
    for q, (fit_idx, hold_idx) in enumerate(split_iter, start=1):
        X_fit = X_train[fit_idx]
        y_fit = y_bin_train[fit_idx]

        model_q = _fit_mfv_model(
            X_fit, y_fit,
            seed=seed + q,
            M=M, rho=rho,
            n_candidates=N_cand,
            r_min=r_min, r_max=r_max,
            N_max=N_max, beta=beta, delta=delta
        )

        # Algorithm 5 line 7: threshold fitted without fold q.
        fit_tmp = {
            **model_q,
            "mfv_centers": model_q["centers"],
            "mfv_radii": model_q["radii"],
            "threshold": 0.5
        }
        fit_scores = mfv_nsa_features_batch(
            X_fit, fit_tmp, tau=0.5,
            gamma=gamma, w_d=w_d, w_a=w_a, eps=eps, K=K
        )[:, 1]
        tau_q, _ = tune_mfv_threshold(y_fit, fit_scores)

        hold_tmp = {
            **model_q,
            "mfv_centers": model_q["centers"],
            "mfv_radii": model_q["radii"],
            "threshold": tau_q
        }
        Z_oof[hold_idx] = mfv_nsa_features_batch(
            X_train[hold_idx], hold_tmp, tau=tau_q,
            gamma=gamma, w_d=w_d, w_a=w_a, eps=eps, K=K
        )

    # Phase 2 — Final Training-Only Immune Model.
    final_model = _fit_mfv_model(
        X_train, y_bin_train,
        seed=seed,
        M=M, rho=rho,
        n_candidates=N_cand,
        r_min=r_min, r_max=r_max,
        N_max=N_max, beta=beta, delta=delta
    )

    # Algorithm 5 line 16: final threshold from training-derived OOF scores.
    tau_star, tau_star_macro_f1 = tune_mfv_threshold(
        y_bin_train, Z_oof[:, 1]
    )

    final_model.update({
        "mfv_centers": final_model["centers"].copy(),
        "mfv_radii": final_model["radii"].copy(),
        "threshold": tau_star,
        "threshold_oof_macro_f1": tau_star_macro_f1
    })

    # Phase 3 — Inductive Evaluation-Feature Generation.
    Z_eval = mfv_nsa_features_batch(
        X_eval, final_model, tau=tau_star,
        gamma=gamma, w_d=w_d, w_a=w_a, eps=eps, K=K
    )

    return Z_oof, Z_eval, final_model


# ============================================================
# Table 5 — Behavior of the distance-based membership function
# ============================================================
TABLE5_MEMBERSHIP_BEHAVIOR = pd.DataFrame([
    {
        "Distance regime": "Near the self-set",
        "Condition": "d_self << r_boundary",
        "Membership": "mu_dist ≈ 0"
    },
    {
        "Distance regime": "At the adaptive boundary",
        "Condition": "d_self = r_boundary",
        "Membership": "mu_dist = 0.5"
    },
    {
        "Distance regime": "Far from the self-set",
        "Condition": "d_self >> r_boundary",
        "Membership": "mu_dist ≈ 1"
    }
])

print("Table 5 — Behavior of the distance-based membership function")
display(TABLE5_MEMBERSHIP_BEHAVIOR)


# ============================================================
# Table 6 — Components of the 14-D MFV-NSA feature vector
# ============================================================
TABLE6_MFV_NSA_FEATURES = pd.DataFrame([
    (1,  "Nearest-self distance",            "d_self",          "mfv_d_self"),
    (2,  "Fuzzy anomaly score",              "s_fuzzy",         "mfv_s_fuzzy"),
    (3,  "Distance membership",              "mu_dist",         "mfv_mu_dist"),
    (4,  "Detector-activation membership",   "mu_det",          "mfv_mu_det"),
    (5,  "Minimum detector distance",        "d_min",           "mfv_d_min"),
    (6,  "Mean detector distance",           "d_mean",          "mfv_d_mean"),
    (7,  "Detector-distance deviation",      "sigma_d",         "mfv_d_std"),
    (8,  "Minimum activation radius",        "nu_min",          "mfv_r_min_active"),
    (9,  "Mean activation radius",           "nu_mean",         "mfv_r_mean_active"),
    (10, "Activation-radius deviation",      "sigma_nu",        "mfv_r_std_active"),
    (11, "Activated-detector count",         "act_K'",          "mfv_act_k"),
    (12, "Maximum detector penetration",     "p_max",           "mfv_p_max"),
    (13, "Mean detector penetration",        "p_mean",          "mfv_p_mean"),
    (14, "Decision margin",                  "s_fuzzy - tau",   "mfv_margin"),
], columns=["No.", "Feature", "Paper symbol", "Notebook column"])

print("Table 6 — Components of the 14-D MFV-NSA Feature Vector")
display(TABLE6_MFV_NSA_FEATURES)


# ============================================================
# FinalV2 compatibility wrapper
# Keeps the original downstream pipeline/cell call unchanged:
#     multi_objective_v_detector_nsa(...)
# but executes the exact MFV-NSA Algorithms 1–4 internally.
# ============================================================
def multi_objective_v_detector_nsa(
    shape_name,
    n_detectors=MFV_N_MAX,
    grid_size=GRID_SIZE
):
    """
    Backward-compatible function name for the original FinalV2 pipeline.

    IMPORTANT:
    This is no longer a multi-objective/Pareto detector. It now fits the
    exact Multi-Constrained Fuzzy V-Detector NSA from Section 3.4.2.
    """
    # Current shape samples are already available before this cell.
    sub = dataset[dataset["shape"] == shape_name].copy()
    X_all = sub[["x", "y"]].to_numpy(dtype=float)

    # Paper binary convention: 0 = benign/self, 1 = attack/non-self.
    y_bin_all = (sub["self_label"].to_numpy(dtype=int) == 0).astype(int)

    # Use an internal training/validation split for fitting the immune model
    # and selecting tau; the evaluation features are transformed afterward.
    X_fit, X_val, y_fit, y_val = train_test_split(
        X_all,
        y_bin_all,
        test_size=0.20,
        random_state=RANDOM_STATE + SHAPES.index(shape_name),
        stratify=y_bin_all
    )

    model = _fit_mfv_model(
        X_fit,
        y_fit,
        seed=RANDOM_STATE + SHAPES.index(shape_name),
        N_max=min(int(n_detectors), MFV_N_MAX)
    )

    tmp_model = {
        **model,
        "mfv_centers": model["centers"],
        "mfv_radii": model["radii"],
        "threshold": 0.5
    }
    val_scores = mfv_nsa_features_batch(X_val, tmp_model, tau=0.5)[:, 1]
    tau, tau_macro_f1 = tune_mfv_threshold(y_val, val_scores)

    centers = model["centers"].copy()
    radii = model["radii"].copy()

    xs, ys, xx, yy, grid_points = make_grid(grid_size)
    self_grid = shape_mask(grid_points, shape_name)
    nonself_grid = ~self_grid

    # Exact detector coverage (for SDS visualization only).
    covered = np.zeros(len(grid_points), dtype=bool)
    for c, r in zip(centers, radii):
        covered |= (
            np.linalg.norm(grid_points - c, axis=1) <= r
        ) & nonself_grid

    result = {
        "shape": shape_name,

        # Working keys retained for original FinalV2 downstream cells.
        "centers": centers.copy(),
        "radii": radii.copy(),
        "scores": np.empty((len(centers),), dtype=float),

        # Frozen exact MFV-NSA keys.
        "mfv_centers": centers.copy(),
        "mfv_radii": radii.copy(),
        "mfv_covered_grid": covered.copy(),
        "self_set": model["self_set"],
        "self_tree": model["self_tree"],
        "boundary_radius": model["boundary_radius"],
        "candidate_pool": model["candidate_pool"],
        "threshold": tau,
        "threshold_validation_macro_f1": tau_macro_f1,

        # Existing visualization keys.
        "grid_points": grid_points,
        "self_grid": self_grid,
        "nonself_grid": nonself_grid,
        "covered_grid": covered.copy(),
        "xx": xx,
        "yy": yy,
        "xs": xs,
        "ys": ys
    }

    return result


# Compatibility name retained for any older cell/user code.
# For exact Section 3.4.2 scoring, pass a full model_result to
# mfv_nsa_score_batch(...).
def fuzzy_nsa_score(points, centers, radii, model_result=None):
    if model_result is None:
        raise ValueError(
            "Exact MFV-NSA fuzzy scoring requires self_tree and r_boundary. "
            "Use mfv_nsa_score_batch(points, model_result)."
        )
    return mfv_nsa_score_batch(points, model_result)


In [ ]:
# ============================================================
# 7. Adaptive Memory CSA
# ============================================================

def detector_quality(center, radius, grid_points, nonself_grid, self_grid, current_covered):
    d_grid = np.linalg.norm(grid_points - center, axis=1)
    region = d_grid <= radius
    valid_nonself = region & nonself_grid

    new_cov = np.sum(valid_nonself & (~current_covered))
    self_intrusion = np.sum(region & self_grid) / (np.sum(region) + 1e-12)
    precision = np.sum(valid_nonself) / (np.sum(region) + 1e-12)
    return new_cov + 50 * precision - 120 * self_intrusion


def adaptive_memory_csa(detector_result, n_epochs=8, clone_factor=4, mutation_scale=0.035):
    """
    Adaptive Memory CSA:
      1. Keep elite memory detectors.
      2. Clone high-quality detectors.
      3. Mutate weak detectors more strongly.
      4. Replace memory using multi-objective detector quality.
    """
    centers = detector_result["centers"].copy()
    radii = detector_result["radii"].copy()
    grid_points = detector_result["grid_points"]
    self_grid = detector_result["self_grid"]
    nonself_grid = detector_result["nonself_grid"]

    memory_size = len(centers)

    for epoch in range(n_epochs):
        covered = np.zeros(len(grid_points), dtype=bool)
        qualities = []

        for c, r in zip(centers, radii):
            q = detector_quality(c, r, grid_points, nonself_grid, self_grid, covered)
            qualities.append(q)
            covered |= (np.linalg.norm(grid_points - c, axis=1) <= r) & nonself_grid

        qualities = np.asarray(qualities)
        order = np.argsort(qualities)[::-1]
        centers = centers[order]
        radii = radii[order]
        qualities = qualities[order]

        # Elite memory
        elite_count = max(5, int(0.25 * memory_size))
        elite_centers = centers[:elite_count]
        elite_radii = radii[:elite_count]

        clone_centers = []
        clone_radii = []

        q_min, q_max = qualities.min(), qualities.max()
        q_norm = (qualities - q_min) / (q_max - q_min + 1e-12)

        for c, r, qn in zip(centers, radii, q_norm):
            # low quality => higher mutation; high quality => smaller mutation
            adaptive_mutation = mutation_scale * (1.25 - qn)

            for _ in range(clone_factor):
                new_c = np.clip(c + rng.normal(0, adaptive_mutation, size=2), 0, 1)
                # keep detector center in non-self region
                if shape_mask(new_c.reshape(1, -1), detector_result["shape"])[0]:
                    continue
                new_r = np.clip(r + rng.normal(0, adaptive_mutation / 2), MIN_RADIUS, MAX_RADIUS)
                clone_centers.append(new_c)
                clone_radii.append(new_r)

        if len(clone_centers) > 0:
            all_centers = np.vstack([elite_centers, np.asarray(clone_centers)])
            all_radii = np.r_[elite_radii, np.asarray(clone_radii)]
        else:
            all_centers = elite_centers
            all_radii = elite_radii

        # Reselect best memory
        covered = np.zeros(len(grid_points), dtype=bool)
        selected_c, selected_r = [], []

        for _ in range(memory_size):
            best_i, best_q = None, -np.inf
            for i in range(len(all_centers)):
                q = detector_quality(all_centers[i], all_radii[i], grid_points, nonself_grid, self_grid, covered)
                q -= 60 * detector_overlap_penalty(all_centers[i], all_radii[i], selected_c, selected_r)
                if q > best_q:
                    best_i, best_q = i, q

            if best_i is None:
                break

            selected_c.append(all_centers[best_i])
            selected_r.append(all_radii[best_i])
            covered |= (np.linalg.norm(grid_points - all_centers[best_i], axis=1) <= all_radii[best_i]) & nonself_grid

            all_centers = np.delete(all_centers, best_i, axis=0)
            all_radii = np.delete(all_radii, best_i, axis=0)
            if len(all_centers) == 0:
                break

        centers = np.asarray(selected_c)
        radii = np.asarray(selected_r)

    detector_result["centers"] = centers
    detector_result["radii"] = radii

    # recompute covered grid
    covered = np.zeros(len(grid_points), dtype=bool)
    for c, r in zip(centers, radii):
        covered |= (np.linalg.norm(grid_points - c, axis=1) <= r) & nonself_grid
    detector_result["covered_grid"] = covered

    return detector_result


In [ ]:
# ============================================================
# 8. Detector metrics and heat-map / hole-map visualization
# ============================================================

def compute_detector_metrics(detector_result):
    grid_points = detector_result["grid_points"]
    nonself_grid = detector_result["nonself_grid"]
    self_grid = detector_result["self_grid"]
    covered = detector_result.get("mfv_covered_grid", detector_result["covered_grid"])
    centers = detector_result.get("mfv_centers", detector_result["centers"])
    radii = detector_result.get("mfv_radii", detector_result["radii"])

    total_nonself = np.sum(nonself_grid)
    covered_nonself = np.sum(covered & nonself_grid)
    holes = nonself_grid & (~covered)
    hole_count = int(np.sum(holes))
    coverage = covered_nonself / (total_nonself + 1e-12)

    # overlap pair rate
    overlap_pairs = 0
    total_pairs = 0
    for i in range(len(centers)):
        for j in range(i + 1, len(centers)):
            total_pairs += 1
            if np.linalg.norm(centers[i] - centers[j]) < (radii[i] + radii[j]):
                overlap_pairs += 1
    overlap_rate = overlap_pairs / (total_pairs + 1e-12)

    # boundary precision: percentage detector cells that stay in non-self
    total_detector_cells = 0
    valid_detector_cells = 0
    for c, r in zip(centers, radii):
        region = np.linalg.norm(grid_points - c, axis=1) <= r
        total_detector_cells += np.sum(region)
        valid_detector_cells += np.sum(region & nonself_grid)
    boundary_precision = valid_detector_cells / (total_detector_cells + 1e-12)

    return {
        "shape": detector_result["shape"],
        "coverage": coverage,
        "hole_count": hole_count,
        "hole_percent": 100 * hole_count / (total_nonself + 1e-12),
        "overlap_rate": overlap_rate,
        "boundary_precision": boundary_precision,
        "n_detectors": len(centers)
    }


def smooth_grid_values(values, grid_size):
    """Small local smoothing without scipy."""
    z = values.reshape(grid_size, grid_size).astype(float)
    padded = np.pad(z, 1, mode="edge")
    out = np.zeros_like(z)
    for dx in range(3):
        for dy in range(3):
            out += padded[dx:dx+grid_size, dy:dy+grid_size]
    return out / 9.0


def plot_heatmap_and_holes(detector_result):
    shape_name = detector_result["shape"]
    grid_points = detector_result["grid_points"]
    self_grid = detector_result["self_grid"]
    nonself_grid = detector_result["nonself_grid"]
    covered = detector_result.get("mfv_covered_grid", detector_result["covered_grid"])
    centers = detector_result.get("mfv_centers", detector_result["centers"])
    radii = detector_result.get("mfv_radii", detector_result["radii"])

    grid_size = detector_result["xx"].shape[0]

    # fuzzy coverage heat-map
    fuzzy_scores = mfv_nsa_score_batch(grid_points, detector_result)
    fuzzy_scores[self_grid] = 0.0
    heat = smooth_grid_values(fuzzy_scores, grid_size)

    holes = nonself_grid & (~covered)
    hole_map = np.full(len(grid_points), 0.95)      # red background
    hole_map[self_grid] = 0.45                      # shape body
    hole_map[holes] = 0.05                          # holes
    hole_map = hole_map.reshape(grid_size, grid_size)

    metrics = compute_detector_metrics(detector_result)

    fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2))

    im0 = axes[0].imshow(
        heat, extent=[0, 1, 0, 1], origin="lower",
        cmap="RdYlGn", vmin=0, vmax=1
    )
    axes[0].contour(
        detector_result["xx"], detector_result["yy"],
        self_grid.reshape(grid_size, grid_size),
        levels=[0.5], colors="black", linewidths=0.5
    )
    axes[0].set_title(f"{shape_name}: coverage heat-map", fontweight="bold", fontsize=10)
    axes[0].set_xlim(0, 1)
    axes[0].set_ylim(0, 1)
    axes[0].set_aspect("equal")
    fig.colorbar(im0, ax=axes[0], fraction=0.046)

    im1 = axes[1].imshow(
        hole_map, extent=[0, 1, 0, 1], origin="lower",
        cmap="RdYlGn_r", vmin=0, vmax=1
    )
    axes[1].contour(
        detector_result["xx"], detector_result["yy"],
        self_grid.reshape(grid_size, grid_size),
        levels=[0.5], colors="black", linewidths=0.5
    )
    axes[1].set_title(
        f"{shape_name}: holes={metrics['hole_count']} ({metrics['hole_percent']:.1f}%)",
        fontweight="bold", fontsize=10
    )
    axes[1].set_xlim(0, 1)
    axes[1].set_ylim(0, 1)
    axes[1].set_aspect("equal")
    fig.colorbar(im1, ax=axes[1], fraction=0.046)

    plt.tight_layout()
    plt.show()

    return metrics


def plot_detectors(detector_result):
    shape_name = detector_result["shape"]
    centers = detector_result.get("mfv_centers", detector_result["centers"])
    radii = detector_result.get("mfv_radii", detector_result["radii"])
    grid_points = detector_result["grid_points"]
    self_grid = detector_result["self_grid"]
    grid_size = detector_result["xx"].shape[0]

    plt.figure(figsize=(5, 5))
    plt.contourf(
        detector_result["xx"], detector_result["yy"],
        self_grid.reshape(grid_size, grid_size),
        levels=[-0.1, 0.5, 1.1], alpha=0.35
    )

    ax = plt.gca()
    for c, r in zip(centers, radii):
        circle = plt.Circle(c, r, fill=False, alpha=0.35, linewidth=0.8)
        ax.add_patch(circle)

    plt.scatter(centers[:, 0], centers[:, 1], s=8, c="red", label="Detectors")
    plt.title(f"{shape_name}: Adaptive Memory CSA Detectors", fontweight="bold")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.gca().set_aspect("equal")
    plt.legend()
    plt.show()


In [ ]:
# ============================================================
# 9. Run NSA + CSA and plot heat-map / hole-map for all shapes
# ============================================================

detector_bank = {}
metric_rows = []

for shp in SHAPES:
    print(f"\nProcessing shape: {shp}")
    result = multi_objective_v_detector_nsa(shp, n_detectors=MFV_N_MAX, grid_size=GRID_SIZE)
    result = adaptive_memory_csa(result, n_epochs=8)
    detector_bank[shp] = result

    metrics = plot_heatmap_and_holes(result)
    metric_rows.append(metrics)

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)


In [ ]:
# ============================================================
# 9A. NSA Performance Curves — All SDS Shapes
#     Metrics vs. self-antigen radius:
#       1. Detection rate (%)
#       2. False alarm rate (%)
#       3. Number of detectors (N)
#       4. Detector train time (s)
# ============================================================

import time

SELF_ANTIGEN_RADII = np.round(np.linspace(0.005, 0.060, 8), 3)
PERF_N_CANDIDATES = 900
PERF_MAX_DETECTORS = N_DETECTORS


def generate_candidate_detectors_with_self_radius(shape_name, self_antigen_radius, n_candidates=PERF_N_CANDIDATES):
    """
    Candidate detector generation for performance-curve analysis.
    self_antigen_radius controls how far detectors stay away from self-antigens.
    A larger value makes detectors more conservative and usually reduces false alarm.
    """
    local_rng = np.random.default_rng(RANDOM_STATE + int(self_antigen_radius * 100000) + SHAPES.index(shape_name) * 1000)

    candidates = local_rng.random((n_candidates * 4, 2))
    candidates = candidates[~shape_mask(candidates, shape_name)][:n_candidates]

    self_points = local_rng.random((n_candidates * 4, 2))
    self_points = self_points[shape_mask(self_points, shape_name)]

    if len(candidates) == 0 or len(self_points) < 20:
        return np.empty((0, 2)), np.empty((0,)), self_points

    self_tree = BallTree(self_points)
    dist, _ = self_tree.query(candidates, k=1)

    # Self-antigen radius is subtracted from the nearest-self distance.
    # This prevents detector intrusion into self space.
    radii = np.clip(dist.ravel() - self_antigen_radius, MIN_RADIUS, MAX_RADIUS)
    keep = radii > MIN_RADIUS

    return candidates[keep], radii[keep], self_points


def train_nsa_for_radius(shape_name, self_antigen_radius, n_detectors=PERF_MAX_DETECTORS):
    start_time = time.time()

    xs, ys, xx, yy, grid_points = make_grid(GRID_SIZE)
    self_grid = shape_mask(grid_points, shape_name)
    nonself_grid = ~self_grid
    centers, radii, self_points = generate_candidate_detectors_with_self_radius(shape_name, self_antigen_radius)

    if len(centers) == 0:
        return {
            "shape": shape_name,
            "self_antigen_radius": self_antigen_radius,
            "centers": np.empty((0, 2)),
            "radii": np.empty((0,)),
            "train_time_sec": time.time() - start_time
        }

    covered = np.zeros(len(grid_points), dtype=bool)
    selected_centers, selected_radii = [], []

    # Greedy selection: maximize uncovered non-self coverage and penalize overlap.
    available_idx = np.arange(len(centers))
    for _ in range(min(n_detectors, len(centers))):
        best_idx = None
        best_score = -np.inf
        subset_size = min(300, len(available_idx))
        subset_idx = np.random.choice(available_idx, size=subset_size, replace=False)

        for idx in subset_idx:
            c = centers[idx]
            r = radii[idx]
            d_grid = np.linalg.norm(grid_points - c, axis=1)
            detector_region = d_grid <= r
            valid_region = detector_region & nonself_grid

            new_coverage = np.sum(valid_region & (~covered))
            overlap_pen = detector_overlap_penalty(c, r, selected_centers, selected_radii)
            score = new_coverage - 60.0 * overlap_pen

            if score > best_score:
                best_score = score
                best_idx = idx

        if best_idx is None:
            break

        selected_centers.append(centers[best_idx])
        selected_radii.append(radii[best_idx])
        covered |= (np.linalg.norm(grid_points - centers[best_idx], axis=1) <= radii[best_idx]) & nonself_grid
        available_idx = available_idx[available_idx != best_idx]

        if len(available_idx) == 0:
            break

    return {
        "shape": shape_name,
        "self_antigen_radius": self_antigen_radius,
        "centers": np.asarray(selected_centers),
        "radii": np.asarray(selected_radii),
        "train_time_sec": time.time() - start_time
    }


def evaluate_nsa_detection(df, shape_name, centers, radii):
    sub = df[df["shape"] == shape_name]
    points = sub[["x", "y"]].values
    y_true_attack = (sub["self_label"].values == 0).astype(int)  # 1 = attack/non-self

    if len(centers) == 0:
        y_pred_attack = np.zeros(len(points), dtype=int)
    else:
        d = np.linalg.norm(points[:, None, :] - centers[None, :, :], axis=2)
        y_pred_attack = np.any(d <= radii[None, :], axis=1).astype(int)

    TP = np.sum((y_true_attack == 1) & (y_pred_attack == 1))
    FN = np.sum((y_true_attack == 1) & (y_pred_attack == 0))
    FP = np.sum((y_true_attack == 0) & (y_pred_attack == 1))
    TN = np.sum((y_true_attack == 0) & (y_pred_attack == 0))

    detection_rate = 100.0 * TP / (TP + FN + 1e-12)
    false_alarm_rate = 100.0 * FP / (FP + TN + 1e-12)

    return detection_rate, false_alarm_rate


performance_rows = []

for shape_name in SHAPES:
    for self_radius in SELF_ANTIGEN_RADII:
        trained = train_nsa_for_radius(shape_name, self_radius)
        dr, far = evaluate_nsa_detection(dataset, shape_name, trained["centers"], trained["radii"])

        performance_rows.append({
            "shape": shape_name,
            "self_antigen_radius": self_radius,
            "detection_rate_percent": dr,
            "false_alarm_rate_percent": far,
            "n_detectors": len(trained["centers"]),
            "detector_train_time_sec": trained["train_time_sec"]
        })

nsa_performance_df = pd.DataFrame(performance_rows)
display(nsa_performance_df)


def plot_nsa_performance_curves(performance_df):
    metrics = [
        ("detection_rate_percent", "Detection rate (%)"),
        ("false_alarm_rate_percent", "False alarm rate (%)"),
        ("n_detectors", "Number of detectors (N)"),
        ("detector_train_time_sec", "Detector train time (s)")
    ]

    for metric_col, y_label in metrics:
        plt.figure(figsize=(7.5, 4.6))
        for shape_name in SHAPES:
            sub = performance_df[performance_df["shape"] == shape_name].sort_values("self_antigen_radius")
            plt.plot(
                sub["self_antigen_radius"],
                sub[metric_col],
                marker="o",
                linewidth=2,
                label=shape_name
            )

        plt.title(f"NSA Performance Curves — All SDS Shapes\n{y_label} vs. self-antigen radius", fontweight="bold")
        plt.xlabel("Self-antigen radius")
        plt.ylabel(y_label)
        plt.grid(alpha=0.30)
        plt.legend()
        plt.tight_layout()
        plt.show()


plot_nsa_performance_curves(nsa_performance_df)


In [ ]:

# ============================================================
# 10. Add EXACT 14-D MFV-NSA features to 10,000-shape datasets
#     (Stage 5 / Algorithm 4 / Table 6)
# ============================================================

def add_fuzzy_nsa_features(df, detector_bank):
    """
    Preserve the original FinalV2 pipeline step, but replace the previous
    3-feature fuzzy NSA output with the paper's exact 14-D MFV-NSA vector.

    The three legacy columns are also retained as aliases so later user code
    that expects them does not break.
    """
    parts = []

    for shp in SHAPES:
        sub = df[df["shape"] == shp].copy()
        points = sub[["x", "y"]].to_numpy(dtype=float)

        # Exact Algorithm 4, vectorized.
        Z_nsa = mfv_nsa_features_batch(
            points,
            detector_bank[shp],
            tau=detector_bank[shp]["threshold"]
        )

        for j, col in enumerate(MFV_NSA_FEATURE_COLUMNS):
            sub[col] = Z_nsa[:, j]

        # Backward-compatible aliases from the original FinalV2 notebook.
        sub["fuzzy_nsa_score"] = sub["mfv_s_fuzzy"]
        sub["detector_hit"] = (sub["mfv_act_k"] > 0).astype(int)
        sub["min_detector_dist"] = sub["mfv_d_min"]

        parts.append(sub)

    return pd.concat(parts, ignore_index=True)


dataset_nsa = add_fuzzy_nsa_features(dataset, detector_bank)

print("MFV-NSA feature columns (Table 6):")
print(MFV_NSA_FEATURE_COLUMNS)
display(dataset_nsa.head())


In [ ]:
# ============================================================
# 11. Final train/test evaluation only
#     No fivefold cross-validation is used.
# ============================================================

feature_cols = [
    "x", "y", "shape_id", "dist_center", "angle", "xy", "x2", "y2",
] + MFV_NSA_FEATURE_COLUMNS

X = dataset_nsa[feature_cols].values
y = dataset_nsa["self_label"].values   # 1 = self, 0 = non-self

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=14,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight="balanced"
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

# ADR for non-self/attack detection:
# attack = non-self = 0
attack_true = (y_test == 0).astype(int)
attack_pred = (y_pred == 0).astype(int)
adr = recall_score(attack_true, attack_pred, zero_division=0)

print("Final Train/Test Evaluation Only")
print("Accuracy :", round(acc, 4))
print("Precision:", round(prec, 4))
print("Recall   :", round(rec, 4))
print("F1-score :", round(f1, 4))
print("ADR      :", round(adr, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Non-Self/Attack", "Self/Normal"]))


In [ ]:
# ============================================================
# 12. Per-shape final evaluation
# ============================================================

eval_rows = []
test_df = pd.DataFrame(X_test, columns=feature_cols)
test_df["y_true"] = y_test
test_df["y_pred"] = y_pred

for shp in SHAPES:
    sid = SHAPES.index(shp)
    sub = test_df[test_df["shape_id"] == sid]
    yt = sub["y_true"].values
    yp = sub["y_pred"].values

    attack_true = (yt == 0).astype(int)
    attack_pred = (yp == 0).astype(int)

    eval_rows.append({
        "Shape": shp,
        "Samples": len(sub),
        "Accuracy": accuracy_score(yt, yp),
        "Precision": precision_score(yt, yp, zero_division=0),
        "Recall": recall_score(yt, yp, zero_division=0),
        "F1": f1_score(yt, yp, zero_division=0),
        "ADR": recall_score(attack_true, attack_pred, zero_division=0)
    })

eval_df = pd.DataFrame(eval_rows)
display(eval_df)


In [ ]:
# ============================================================
# 13. Save generated dataset and metrics
# ============================================================

dataset_nsa.to_csv("SDS_shapes_10000_self_nonself_with_fuzzy_NSA.csv", index=False)
metrics_df.to_csv("NSA_CSA_detector_quality_metrics.csv", index=False)
eval_df.to_csv("final_train_test_results_no_cv.csv", index=False)

print("Saved:")
print("1. SDS_shapes_10000_self_nonself_with_fuzzy_NSA.csv")
print("2. NSA_CSA_detector_quality_metrics.csv")
print("3. final_train_test_results_no_cv.csv")


In [ ]:
# Save NSA performance curve results
nsa_performance_df.to_csv("NSA_performance_curves_all_SDS_shapes.csv", index=False)
print("Saved: NSA_performance_curves_all_SDS_shapes.csv")


# Added Graphs and Coverage Maps

The following cells are appended without changing the original code. They add:

1. Bar graphs for coverage score, hole percent, and overlap rate.
2. Threshold curves for detection rate, false positive rate, and F1 score.
3. Self-antigen-number curves for detection rate, false positive rate, and detector training time.
4. Non-self coverage maps for all SDS datasets.


In [ ]:

# ============================================================
# 14A. Bar Graphs: Coverage Score, Hole Percent, and Overlap Rate
#      This cell uses the existing metrics_df generated by the original code.
# ============================================================

def plot_detector_quality_bar_graphs(metrics_df):
    plot_df = metrics_df.copy()

    # Keep SDS shape order consistent
    plot_df["shape"] = pd.Categorical(plot_df["shape"], categories=SHAPES, ordered=True)
    plot_df = plot_df.sort_values("shape")

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    fig.suptitle("Detector Quality Metrics — All SDS Shapes", fontsize=15, fontweight="bold")

    bar_colors = ["#E76F51", "#4C9BE8", "#2CA58D", "#9B59B6"]

    # 1. Coverage score
    axes[0].bar(plot_df["shape"], plot_df["coverage"], color=bar_colors, edgecolor="black", alpha=0.90)
    axes[0].set_title("Coverage Score (higher=better)", fontweight="bold")
    axes[0].set_ylabel("Coverage Score")
    axes[0].set_ylim(0, 1.05)
    axes[0].grid(axis="y", alpha=0.35)
    for i, v in enumerate(plot_df["coverage"]):
        axes[0].text(i, v + 0.01, f"{v:.4f}", ha="center", fontsize=9)

    # 2. Hole percent
    axes[1].bar(plot_df["shape"], plot_df["hole_percent"], color=bar_colors, edgecolor="black", alpha=0.90)
    axes[1].set_title("Hole Percent (lower=better)", fontweight="bold")
    axes[1].set_ylabel("Hole Percent (%)")
    axes[1].grid(axis="y", alpha=0.35)
    for i, v in enumerate(plot_df["hole_percent"]):
        axes[1].text(i, v + max(plot_df["hole_percent"]) * 0.02, f"{v:.4f}", ha="center", fontsize=9)

    # 3. Overlap rate
    axes[2].bar(plot_df["shape"], plot_df["overlap_rate"], color=bar_colors, edgecolor="black", alpha=0.90)
    axes[2].set_title("Overlap Rate (lower=better)", fontweight="bold")
    axes[2].set_ylabel("Overlap Rate")
    axes[2].grid(axis="y", alpha=0.35)
    for i, v in enumerate(plot_df["overlap_rate"]):
        axes[2].text(i, v + max(plot_df["overlap_rate"]) * 0.03, f"{v:.4f}", ha="center", fontsize=9)

    plt.tight_layout()
    plt.show()


plot_detector_quality_bar_graphs(metrics_df)


In [ ]:

# ============================================================
# 14B. Threshold Curves:
#      Detection Rate (%) vs Threshold
#      False Positive Rate (%) vs Threshold
#      F1 Score vs Threshold
# ============================================================

def threshold_curve_metrics(df, detector_bank, thresholds=np.round(np.linspace(0.05, 0.95, 10), 2)):
    rows = []

    for shape_name in SHAPES:
        sub = df[df["shape"] == shape_name].copy()
        points = sub[["x", "y"]].values
        y_attack_true = (sub["self_label"].values == 0).astype(int)  # 1 = non-self/attack

        fuzzy_scores = mfv_nsa_score_batch(points, detector_bank[shape_name])

        for th in thresholds:
            y_attack_pred = (fuzzy_scores >= th).astype(int)

            TP = np.sum((y_attack_true == 1) & (y_attack_pred == 1))
            FN = np.sum((y_attack_true == 1) & (y_attack_pred == 0))
            FP = np.sum((y_attack_true == 0) & (y_attack_pred == 1))
            TN = np.sum((y_attack_true == 0) & (y_attack_pred == 0))

            detection_rate = 100.0 * TP / (TP + FN + 1e-12)
            false_positive_rate = 100.0 * FP / (FP + TN + 1e-12)
            precision = TP / (TP + FP + 1e-12)
            recall = TP / (TP + FN + 1e-12)
            f1 = 2.0 * precision * recall / (precision + recall + 1e-12)

            rows.append({
                "shape": shape_name,
                "threshold": th,
                "detection_rate_percent": detection_rate,
                "false_positive_rate_percent": false_positive_rate,
                "f1_score": f1
            })

    return pd.DataFrame(rows)


threshold_metrics_df = threshold_curve_metrics(dataset, detector_bank)
display(threshold_metrics_df.head())


def plot_threshold_curves(threshold_df):
    curve_specs = [
        ("detection_rate_percent", "Detection Rate (%)", "Detection Rate (%) vs. Threshold Curve"),
        ("false_positive_rate_percent", "False Positive Rate (%)", "False Positive Rate (%) vs. Threshold Curve"),
        ("f1_score", "F1 Score", "F1 Score vs. Threshold Curve")
    ]

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

    for ax, (metric_col, y_label, title) in zip(axes, curve_specs):
        for shape_name in SHAPES:
            sub = threshold_df[threshold_df["shape"] == shape_name].sort_values("threshold")
            ax.plot(sub["threshold"], sub[metric_col], marker="o", linewidth=2, label=shape_name)

        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Threshold theta")
        ax.set_ylabel(y_label)
        ax.grid(alpha=0.35)
        ax.legend()

    plt.tight_layout()
    plt.show()


plot_threshold_curves(threshold_metrics_df)


In [ ]:

# ============================================================
# 14C. NSA Performance Curves vs Self-Antigens Number (N):
#      Detection Rate (%) vs Self-Antigens Number
#      False Positive Rate (%) vs Self-Antigens Number
#      Detector Training Time (s) vs Self-Antigens Number
# ============================================================

SELF_ANTIGEN_NUMBERS = [250, 500, 1000, 1500, 2000, 2500]


def train_nsa_with_self_antigen_number(shape_name, n_self_antigens, n_candidates=PERF_N_CANDIDATES, n_detectors=PERF_MAX_DETECTORS):
    start_time = time.time()

    xs, ys, xx, yy, grid_points = make_grid(GRID_SIZE)
    self_grid = shape_mask(grid_points, shape_name)
    nonself_grid = ~self_grid

    local_rng = np.random.default_rng(RANDOM_STATE + SHAPES.index(shape_name) * 2000 + int(n_self_antigens))

    # Candidate detectors are generated in non-self space.
    candidates = local_rng.random((n_candidates * 5, 2))
    candidates = candidates[~shape_mask(candidates, shape_name)][:n_candidates]

    # Self antigens are generated inside the shape.
    self_pool = local_rng.random((max(n_self_antigens * 10, 5000), 2))
    self_pool = self_pool[shape_mask(self_pool, shape_name)]

    if len(self_pool) == 0 or len(candidates) == 0:
        return {
            "shape": shape_name,
            "self_antigens_number": n_self_antigens,
            "centers": np.empty((0, 2)),
            "radii": np.empty((0,)),
            "train_time_sec": time.time() - start_time
        }

    n_take = min(n_self_antigens, len(self_pool))
    self_points = self_pool[local_rng.choice(len(self_pool), size=n_take, replace=False)]

    self_tree = BallTree(self_points)
    dist, _ = self_tree.query(candidates, k=1)
    radii = np.clip(dist.ravel(), MIN_RADIUS, MAX_RADIUS)

    keep = radii > MIN_RADIUS
    centers = candidates[keep]
    radii = radii[keep]

    covered = np.zeros(len(grid_points), dtype=bool)
    selected_centers, selected_radii = [], []

    available_idx = np.arange(len(centers))
    for _ in range(min(n_detectors, len(centers))):
        best_idx = None
        best_score = -np.inf

        subset_size = min(300, len(available_idx))
        subset_idx = local_rng.choice(available_idx, size=subset_size, replace=False)

        for idx in subset_idx:
            c = centers[idx]
            r = radii[idx]

            d_grid = np.linalg.norm(grid_points - c, axis=1)
            detector_region = d_grid <= r
            valid_region = detector_region & nonself_grid

            new_coverage = np.sum(valid_region & (~covered))
            overlap_pen = detector_overlap_penalty(c, r, selected_centers, selected_radii)
            score = new_coverage - 60.0 * overlap_pen

            if score > best_score:
                best_score = score
                best_idx = idx

        if best_idx is None:
            break

        selected_centers.append(centers[best_idx])
        selected_radii.append(radii[best_idx])

        covered |= (np.linalg.norm(grid_points - centers[best_idx], axis=1) <= radii[best_idx]) & nonself_grid
        available_idx = available_idx[available_idx != best_idx]

        if len(available_idx) == 0:
            break

    return {
        "shape": shape_name,
        "self_antigens_number": n_self_antigens,
        "centers": np.asarray(selected_centers),
        "radii": np.asarray(selected_radii),
        "train_time_sec": time.time() - start_time
    }


self_number_rows = []

for shape_name in SHAPES:
    for n_self in SELF_ANTIGEN_NUMBERS:
        trained = train_nsa_with_self_antigen_number(shape_name, n_self)
        dr, far = evaluate_nsa_detection(dataset, shape_name, trained["centers"], trained["radii"])

        self_number_rows.append({
            "shape": shape_name,
            "self_antigens_number": n_self,
            "detection_rate_percent": dr,
            "false_positive_rate_percent": far,
            "detector_train_time_sec": trained["train_time_sec"],
            "n_detectors": len(trained["centers"])
        })

self_antigen_number_df = pd.DataFrame(self_number_rows)
display(self_antigen_number_df.head())


def plot_self_antigen_number_curves(curve_df):
    curve_specs = [
        ("detection_rate_percent", "Detection Rate (%)", "Detection Rate (%) vs. Self-Antigens Number"),
        ("false_positive_rate_percent", "False Positive Rate (%)", "False Positive Rate (%) vs. Self-Antigens Number"),
        ("detector_train_time_sec", "Detector Training Time (s)", "Detector Training Time (s) vs. Self-Antigens Number")
    ]

    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

    for ax, (metric_col, y_label, title) in zip(axes, curve_specs):
        for shape_name in SHAPES:
            sub = curve_df[curve_df["shape"] == shape_name].sort_values("self_antigens_number")
            ax.plot(sub["self_antigens_number"], sub[metric_col], marker="o", linewidth=2, label=shape_name)

        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Self-antigens number (N)")
        ax.set_ylabel(y_label)
        ax.grid(alpha=0.35)
        ax.legend()

    plt.tight_layout()
    plt.show()


plot_self_antigen_number_curves(self_antigen_number_df)


In [ ]:

# ============================================================
# 14D. Non-Self Coverage Map for All SDS Datasets
#      Green = covered non-self region
#      Red = uncovered non-self hole
#      Light area = self region
# ============================================================

def plot_nonself_coverage_maps(detector_bank):
    fig, axes = plt.subplots(2, 2, figsize=(10, 9))
    axes = axes.ravel()

    for ax, shape_name in zip(axes, SHAPES):
        result = detector_bank[shape_name]

        grid_points = result["grid_points"]
        grid_size = result["xx"].shape[0]
        self_grid = result["self_grid"]
        nonself_grid = result["nonself_grid"]
        covered = result.get("mfv_covered_grid", result["covered_grid"])

        coverage_map = np.zeros(len(grid_points))
        coverage_map[self_grid] = 0.50                         # self body
        coverage_map[nonself_grid & covered] = 1.00            # covered non-self
        coverage_map[nonself_grid & (~covered)] = 0.05         # holes

        metrics = compute_detector_metrics(result)

        im = ax.imshow(
            coverage_map.reshape(grid_size, grid_size),
            extent=[0, 1, 0, 1],
            origin="lower",
            cmap="RdYlGn",
            vmin=0,
            vmax=1
        )

        ax.contour(
            result["xx"], result["yy"],
            self_grid.reshape(grid_size, grid_size),
            levels=[0.5],
            colors="black",
            linewidths=0.6
        )

        ax.set_title(
            f"{shape_name}: coverage={metrics['coverage']:.4f}, holes={metrics['hole_count']}",
            fontweight="bold",
            fontsize=10
        )
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_aspect("equal")

    fig.suptitle("Non-Self Coverage Maps — All SDS Datasets", fontsize=14, fontweight="bold")
    fig.colorbar(im, ax=axes.tolist(), shrink=0.75, label="Coverage intensity")
    plt.tight_layout()
    plt.show()


plot_nonself_coverage_maps(detector_bank)


In [ ]:

# ============================================================
# 14E. Save Added Curve Data
# ============================================================

threshold_metrics_df.to_csv("NSA_threshold_curves_all_SDS_shapes.csv", index=False)
self_antigen_number_df.to_csv("NSA_self_antigen_number_curves_all_SDS_shapes.csv", index=False)

print("Saved:")
print("1. NSA_threshold_curves_all_SDS_shapes.csv")
print("2. NSA_self_antigen_number_curves_all_SDS_shapes.csv")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# Data
# =========================
N = [250, 500, 1000, 1500, 2000, 2500]

cross = [99.934, 99.660, 99.856, 99.856, 99.830, 99.594]
ring = [100.000, 100.000, 99.972, 99.943, 99.972, 99.832]
pentagram = [99.952, 99.929, 99.929, 99.822, 99.750, 99.715]
triangle = [100.000, 99.960, 99.947, 99.974, 99.974, 99.947]

# =========================
# Figure (same style)
# =========================
plt.style.use('default')

fig, ax = plt.subplots(figsize=(5.86, 4.68), dpi=100)

ax.plot(
    N, cross,
    marker='o',
    linewidth=2.0,
    markersize=6,
    label='Cross'
)

ax.plot(
    N, ring,
    marker='o',
    linewidth=2.0,
    markersize=6,
    label='Ring'
)

ax.plot(
    N, pentagram,
    marker='o',
    linewidth=2.0,
    markersize=6,
    label='Pentagram'
)

ax.plot(
    N, triangle,
    marker='o',
    linewidth=2.0,
    markersize=6,
    label='Triangle'
)

# =========================
# Labels & Title
# =========================
ax.set_title(
    'Detection Rate (%) vs. Self-Antigens Number',
    fontsize=14,
    fontweight='bold'
)

ax.set_xlabel(
    'Self-antigens number (N)',
    fontsize=12
)

ax.set_ylabel(
    'Detection Rate (%)',
    fontsize=12
)

# =========================
# Y-axis from 98 to 100.5
# =========================
ax.set_ylim(98, 100.5)

ax.set_yticks([
    98.0,
    98.5,
    99.0,
    99.5,
    100.0
])
# Same grid style
ax.grid(True, alpha=0.3)

# Same legend position
ax.legend(
    loc='lower center',
    fontsize=10,
    frameon=True
)

plt.tight_layout()

plt.savefig(
    "Detection_Rate_95_100.png",
    dpi=600,
    bbox_inches='tight'
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# Data
# =========================
N = [250, 500, 1000, 1500, 2000, 2500]

cross = [22.91, 23.75, 24.07, 23.79, 24.87, 23.57]
ring = [23.28, 23.79, 23.74, 23.70, 23.64, 22.92]
pentagram = [23.78, 23.71, 24.05, 23.75, 23.19, 23.47]
triangle = [23.68, 23.73, 23.74, 23.12, 23.36, 23.69]

# =========================
# Figure Style
# =========================
plt.style.use('default')

fig, ax = plt.subplots(
    figsize=(6.2, 4.8),
    dpi=100
)

# =========================
# Plot Curves
# =========================
ax.plot(
    N,
    cross,
    marker='o',
    linewidth=2.5,
    markersize=8,
    label='Cross'
)

ax.plot(
    N,
    ring,
    marker='o',
    linewidth=2.5,
    markersize=8,
    label='Ring'
)

ax.plot(
    N,
    pentagram,
    marker='o',
    linewidth=2.5,
    markersize=8,
    label='Pentagram'
)

ax.plot(
    N,
    triangle,
    marker='o',
    linewidth=2.5,
    markersize=8,
    label='Triangle'
)

# =========================
# Title
# =========================
ax.set_title(
    'Detector Training Time (s) vs. Self-Antigens Number',
    fontsize=20,
    fontweight='bold'
)

# =========================
# Axis Labels
# =========================
ax.set_xlabel(
    'Self-antigens number (N)',
    fontsize=15
)

ax.set_ylabel(
    'Training Time (s)',
    fontsize=15
)

# =========================
# X Axis
# =========================
ax.set_xticks(N)

# =========================
# Y Axis (21–25)
# =========================
ax.set_ylim(20, 26)

ax.set_yticks(
    np.arange(20, 26.1, 0.7)
)

# =========================
# Tick Font Sizes
# =========================
ax.tick_params(
    axis='both',
    labelsize=12
)

# =========================
# Grid (same as attached image)
# =========================
ax.set_axisbelow(True)

ax.grid(
    True,
    which='major',
    axis='both',
    color='#b0b0b0',
    linestyle='-',
    linewidth=1,
    alpha=0.4
)

# =========================
# Legend
# =========================
ax.legend(
    loc='upper left',
    fontsize=12,
    frameon=True
)

# =========================
# Layout
# =========================
plt.tight_layout()

# =========================
# Save Figure
# =========================
plt.savefig(
    'Detector_Training_Time_vs_SelfAntigens.png',
    dpi=800,
    bbox_inches='tight'
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# Data
# ==========================================================
N = [250, 500, 1000, 1500, 2000, 2500]

cross = [22.91, 23.75, 24.07, 23.79, 24.87, 23.57]
ring = [23.28, 23.79, 23.74, 23.70, 23.64, 22.92]
pentagram = [23.78, 23.71, 24.05, 23.75, 23.19, 23.47]
triangle = [23.68, 23.73, 23.74, 23.12, 23.36, 23.69]

# ==========================================================
# Figure Style
# ==========================================================
plt.style.use('default')

fig, ax = plt.subplots(
    figsize=(8, 6),
    dpi=100
)

# ==========================================================
# Plot Curves
# ==========================================================
ax.plot(
    N,
    cross,
    marker='o',
    linewidth=2.5,
    markersize=8,
    label='Cross'
)

ax.plot(
    N,
    ring,
    marker='o',
    linewidth=2.5,
    markersize=8,
    label='Ring'
)

ax.plot(
    N,
    pentagram,
    marker='o',
    linewidth=2.5,
    markersize=8,
    label='Pentagram'
)

ax.plot(
    N,
    triangle,
    marker='o',
    linewidth=2.5,
    markersize=8,
    label='Triangle'
)

# ==========================================================
# Title
# ==========================================================
ax.set_title(
    'Detector Training Time (s) vs. Self-Antigens Number',
    fontsize=18,
    fontweight='bold'
)

# ==========================================================
# Axis Labels
# ==========================================================
ax.set_xlabel(
    'Self-antigens number (N)',
    fontsize=14
)

ax.set_ylabel(
    'Training Time (s)',
    fontsize=14
)

# ==========================================================
# X-axis
# ==========================================================
ax.set_xticks(N)

# ==========================================================
# Y-axis
# ==========================================================
ax.set_ylim(20, 26)

ax.set_yticks(np.arange(20, 26.1, 0.7))

# ==========================================================
# Tick Labels
# ==========================================================
ax.tick_params(
    axis='both',
    labelsize=12
)

# ==========================================================
# Grid (same as attached image)
# ==========================================================
ax.set_axisbelow(True)

ax.grid(
    True,
    which='major',
    axis='both',
    color='#b0b0b0',
    linestyle='-',
    linewidth=1,
    alpha=0.4
)

# ==========================================================
# Legend
# ==========================================================
ax.legend(
    loc='upper left',
    fontsize=12,
    frameon=True
)

# ==========================================================
# Layout
# ==========================================================
plt.tight_layout()

# ==========================================================
# Save Figure
# ==========================================================
plt.savefig(
    'Detector_Training_Time_vs_SelfAntigens.png',
    dpi=600,
    bbox_inches='tight'
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# Data
# ==========================================================
radius = [0.005, 0.013, 0.021, 0.029, 0.036, 0.044, 0.052, 0.060]

cross = [24.38, 24.46, 24.13, 23.36, 24.17, 24.15, 24.20, 24.10]
ring = [23.99, 23.26, 24.00, 24.06, 24.25, 24.77, 25.18, 24.91]
pentagram = [25.80, 24.50, 24.50, 24.33, 24.31, 24.41, 24.30, 23.68]
triangle = [23.76, 24.20, 24.22, 24.28, 24.14, 23.56, 23.63, 24.09]

# ==========================================================
# Figure
# ==========================================================
plt.figure(figsize=(8, 5), dpi=100)

# ==========================================================
# Curves
# ==========================================================
plt.plot(
    radius,
    cross,
    marker='o',
    linewidth=2.2,
    markersize=6,
    label='Cross'
)

plt.plot(
    radius,
    ring,
    marker='o',
    linewidth=2.2,
    markersize=6,
    label='Ring'
)

plt.plot(
    radius,
    pentagram,
    marker='o',
    linewidth=2.2,
    markersize=6,
    label='Pentagram'
)

plt.plot(
    radius,
    triangle,
    marker='o',
    linewidth=2.2,
    markersize=6,
    label='Triangle'
)

# ==========================================================
# Title
# ==========================================================
plt.title(
    'Detector train time (s) vs. self-antigen radius',
    fontsize=17,
    fontweight='bold'
)

# ==========================================================
# Labels
# ==========================================================
plt.xlabel(
    'Self-antigen radius',
    fontsize=12
)

plt.ylabel(
    'Detector train time (s)',
    fontsize=12
)

# ==========================================================
# Y-axis (20–27)
# ==========================================================
plt.ylim(20, 27)

plt.yticks(
    np.arange(20, 27.5, 0.8)
)

# ==========================================================
# Tick sizes
# ==========================================================
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)

# ==========================================================
# Grid (same as figure)
# ==========================================================
plt.grid(
    True,
    color='#b0b0b0',
    linestyle='-',
    linewidth=1,
    alpha=0.35
)

# ==========================================================
# Legend (top center)
# ==========================================================
plt.legend(
    loc='upper center',
    fontsize=11,
    frameon=True
)

plt.tight_layout()

# ==========================================================
# Save
# ==========================================================
plt.savefig(
    'Detector_Training_Time_vs_Radius.png',
    dpi=600,
    bbox_inches='tight'
)

plt.show()